# Single

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

from eval import evaluate_object_detector, parse_and_plot

from pathlib import Path
import pandas as pd

from tqdm import tqdm
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import seaborn as sns

# Set Seaborn style for a more professional look
def set_style():
    sns.set(style="whitegrid", context="notebook")

    plt.rcParams['font.family'] = 'STIXGeneral'
    # Set the DPI for the plots
    plt.rcParams['figure.dpi'] = 220
    
    fontSize = 14
    # Update Matplotlib rcParams for font size
    plt.rcParams.update({
        'font.size': fontSize,
        'axes.titlesize': fontSize,
        'axes.labelsize': fontSize,
        'xtick.labelsize': fontSize,
        'ytick.labelsize': fontSize,
        'legend.fontsize': fontSize,
        'figure.titlesize': fontSize
    })

    # Update Seaborn context with font size settings
    sns.set_context("paper", rc={
        "font.size": fontSize,
        "axes.titlesize": fontSize,
        "axes.labelsize": fontSize,
        "xtick.labelsize": fontSize,
        "ytick.labelsize": fontSize,
        "legend.fontsize": fontSize,
        "figure.titlesize": fontSize
    })

set_style()


    

def find_json(folder, mode='train'):
    jsons = list(folder.glob('**/**/*.json'))
    if mode == 'train':
        # filter out the vis_data folder
        jsons = [j for j in jsons if 'vis_data' in str(j)]
        # not pick the scalars json
        jsons = [j for j in jsons if 'scalars' not in str(j)]
    else:
        # filter out the vis_data folder
        jsons = [j for j in jsons if 'vis_data' not in str(j)]
        # pick the test_results forlder
        jsons = [j for j in jsons if 'coco_metrics' in str(j)]

    assert len(jsons) == 1, f"Found {len(jsons)} json files. Expected 1. \n {jsons}"
    return jsons[0]




set_style()
VISUALIZE = False
# usage
bandSelection = [1]
goTo = ''.join(['_b'+str(i) for i in bandSelection])
band_path = f'/Data_large/marine/PythonProjects/MMDET/checkpoints/VENuS/Single/perfect{goTo}'
folders = list(Path(f'{band_path}').iterdir())
print('Num of folders:', len(folders))

bandsResults = {x:[] for x in range(1,8)}
folders

# for folder in tqdm(folders):
#     try:
#         bandsResults[bandSelection[0]].append(parse_and_plot(find_json(Path(folder), mode='train'), visualize=VISUALIZE))
#     except Exception as e:
#         print(f"Error in folder {folder}: {e}")

In [ ]:
# bandsResults = {x:[] for x in range(1,8)}
def band_parse_testing(json_file_path):
    partial = {
        'Seed': Path(json_file_path).parent.parent.name.split('_')[0],
        'BS': Path(json_file_path).parent.parent.name.split('_')[2],
        'LR': Path(json_file_path).parent.parent.name.split('_')[4],
        'ME': Path(json_file_path).parent.parent.name.split('_')[6],
        'OPT': Path(json_file_path).parent.parent.name.split('_')[8],
    }

    # Read the JSON file
    with open(json_file_path, 'r') as f:
        data = json.load(f)


    # merge data and partial 
    data = {**data, **partial}
    return data

data_x_band = {i:[] for i in range(1,13)}
max_band = 12

for idx in tqdm(range(1,max_band+1)):
    bandSelection = [idx]
    goTo = ''.join(['_b'+str(i) for i in bandSelection])
    band_path = f'/Data_large/marine/PythonProjects/MMDET/checkpoints/VENuS/Single/perfect{goTo}'
    folders = list(Path(f'{band_path}').iterdir())
    for folder in folders:
        try:
            partial = band_parse_testing(find_json(Path(folder), mode='test'))
        except Exception as e:
            print(f"Error in folder {folder}: {e}")
        data_x_band[idx].append(partial)

In [ ]:
all_band = pd.DataFrame()

for idx_band in range(1,13):
    partial = pd.DataFrame(data_x_band[idx_band])
    partial['Band'] = idx_band
    # concat
    all_band = pd.concat([all_band, partial])

all_band

In [ ]:
grouped_bs_lr_me = all_band.groupby(['Band', 'BS', 'LR', 'ME']).mean() # average over seeds
grouped_bs_lr_me

In [ ]:
grouped_bs_lr_me.loc[1].sort_values('coco/bbox_mAP_50', ascending=False)

In [ ]:
grouped_bs_lr_me = all_band.groupby(['Band', 'BS', 'LR', 'ME']).mean() # average over seeds
# take the best BS and LR for EACH BAND in term of coco/bbox_mAP_50
bestConfig = {idx:[] for idx in range(1,13)}    
for idx in range(1,13):
    band = idx
    BS, LR, ME = grouped_bs_lr_me.loc[band]['coco/bbox_mAP_50'].idxmax()
    BS, LR, ME = int(BS), float(LR), int(ME)
    bestConfig[idx] = (BS, LR, ME)

In [ ]:
for key, value in bestConfig.items():
    print(f"Band {key}: BS: {value[0]}, LR: {value[1]}, ME: {value[2]}")

#### Process PR-Curve for each configuration (Run 1 Time)

Running the PR estimations

In [ ]:
pr_x_band = {i:[] for i in range(1,13)}

for idx, value in tqdm(bestConfig.items()):
    print(f"Band: {idx}, BS: {value[0]}, LR: {value[1]}")
    BS = value[0]
    LR = value[1]
    ME = value[2]
    
    Band = idx
    logger.info(f"Band: {Band} \n")
    ann_file = f'/Data_large/marine/Datasets/VENuS/annotations/perfect/test__band_{Band}.json'
    # get folder :
    bandSelection = [Band]
    goTo = ''.join(['_b'+str(i) for i in bandSelection])
    band_path = f'/Data_large/marine/PythonProjects/MMDET/checkpoints/VENuS/Single/perfect{goTo}'
    folders = list(Path(f'{band_path}').iterdir())  
    # filter folders with the BS
    folders = [f for f in folders if (f'BS_{BS}' in f.name) and (f'LR_{LR}' in f.name) and (f'ME_{ME}' in f.name)]
    
    for folder in folders:
        res_file = list(folder.glob('**/*.json'))
        res_file = [x for x in res_file if 'test_results' in str(x)]
        logger.info(f"Folder: {folder}")
        logger.info(f"      Res File: {res_file}")
        try:
            assert len(res_file) == 1, f"Found {len(res_file)} json files. Expected 1. \n {res_file}"
        except AssertionError:
            logger.error(f"Error in {res_file}")
            break
        res_file = res_file[0].as_posix()
        print(f"Res file: {res_file}")
        try:
            stats, eval_results, cocoeval_tmp = evaluate_object_detector(res_file, ann_file)
            pr_x_band[Band].append(cocoeval_tmp)
        except IndexError:
            logger.error(f"Error in {res_file}")
            continue

#### Save

In [ ]:
pd.to_pickle(pr_x_band, 'VENuS/Venus_PR_curve.pkl')

In [ ]:
all_band_best = pd.DataFrame()
for Band in range(1,13):
    BS, LR, ME = bestConfig[Band]
    filtered_all_band = all_band[
        (all_band['Band'] == Band) &
        (all_band['BS'] == str(BS)) &
        (all_band['LR'] == str(LR)) &
        (all_band['ME'] == str(ME))]
    all_band_best = pd.concat([all_band_best, filtered_all_band])

In [ ]:
all_band_best.groupby(['Band','LR','BS','ME']).mean().to_pickle('VENuS/table_Single.pkl')
all_band_best.groupby(['Band','LR','BS','ME']).mean()

### Plots PR and MAP for each configuration

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import pandas as pd

def set_style():

    plt.rcParams['font.family'] = 'STIXGeneral'
    # Set the DPI for the plots
    plt.rcParams['figure.dpi'] = 400
    
    scale_factor = 1.5
    fontSize = 13 * scale_factor
    # Update Matplotlib rcParams for font size
    plt.rcParams.update({
        'font.size': fontSize,
        'axes.titlesize': fontSize,
        'axes.labelsize': fontSize,
        'xtick.labelsize': fontSize,
        'ytick.labelsize': fontSize,
        'legend.fontsize': fontSize,
        'figure.titlesize': fontSize
    })

    # Update Seaborn context with font size settings
    sns.set_context("paper", rc={
        "font.size": fontSize,
        "axes.titlesize": fontSize,
        "axes.labelsize": fontSize,
        "xtick.labelsize": fontSize,
        "ytick.labelsize": fontSize,
        "legend.fontsize": fontSize,
        "figure.titlesize": fontSize
    })
    
set_style()

# Assume pr_x_band and all_band are pre-defined dataframes/lists
pr_x_band = pd.read_pickle('VENuS/Venus_PR_curve.pkl')
# pr_x_band: List of evaluation results for each band
# all_band: DataFrame containing coco/bbox mAP metrics grouped by spectral band

# def plot_precision_recall(ax, pr_x_band):
#     """
#     Helper function to plot precision-recall curves with shaded areas for standard deviation.

#     Parameters:
#     ax (matplotlib.axes._subplots.AxesSubplot): The subplot axes to plot on.
#     pr_x_band (list): A list of evaluation results for different spectral bands.
#     """
#     V = [pr_x_band[i] for i in range(1, 10)]
#     coco_eval_lists = V
#     labels = [f"$B_{{{i}}}$" for i in range(1, 13)]

#     colors = sns.color_palette("colorblind", len(labels)) 
#     for i, coco_eval_list in enumerate(coco_eval_lists):
#         # Extract precision values and recall
#         all_precisions = []
#         recall = np.arange(0.0, 1.01, 0.01)

#         for coco_eval in coco_eval_list:
#             precision = coco_eval.eval['precision'][0, :, 0, 0, 2]  # precision for IoU=0.50:0.95 and area=all
#             all_precisions.append(precision)

#         # Convert list of all precisions to a numpy array for easier manipulation
#         all_precisions = np.array(all_precisions)

#         # Compute the mean precision and the standard deviation for shading
#         mean_precision = np.mean(all_precisions, axis=0)
#         std_precision = np.std(all_precisions, axis=0)

#         upper_bound = mean_precision + std_precision
#         lower_bound = mean_precision - std_precision

#         # Plot shaded area and mean precision line
#         # ax.fill_between(recall, lower_bound, upper_bound, color=colors[i], alpha=0.3)
#         ax.plot(recall, mean_precision, label=labels[i], color=colors[i])

#     ax.set_xlabel('Recall')
#     ax.set_ylabel('Precision')
#     ax.set_ylim([0,1.1])
#     ax.set_xlim([0,1.2])
#     # ax.set_title('Precision-Recall Curves by Band')
#     ax.legend()
#     ax.grid(True)


def plot_precision_recall(ax, pr_x_band):
    """
    Helper function to plot precision-recall curves with shaded areas for standard deviation.

    Parameters:
    ax (matplotlib.axes._subplots.AxesSubplot): The subplot axes to plot on.
    pr_x_band (list): A list of evaluation results for different spectral bands.
    """
    V = [pr_x_band[i] for i in range(1, 13)]
    coco_eval_lists = V
    labels = [f"$B_{{{i}}}$" for i in range(1, 13)]

    colors = sns.color_palette("colorblind", len(labels)) 
    recall_list = []
    lower_bound_list = []
    upper_bound_list = []
    mean_precision_list = []
    
    for i, coco_eval_list in enumerate(coco_eval_lists):
        # Extract precision values and recall
        all_precisions = []
        recall = np.arange(0.0, 1.01, 0.01)
        recall_list.append(recall)

        for coco_eval in coco_eval_list:
            precision = coco_eval.eval['precision'][0, :, 0, 0, 2]  # precision for IoU=0.50:0.95 and area=all
            all_precisions.append(precision)

        # Convert list of all precisions to a numpy array for easier manipulation
        all_precisions = np.array(all_precisions)

        # Compute the mean precision and the standard deviation for shading
        mean_precision = np.mean(all_precisions, axis=0)
        mean_precision_list.append(mean_precision)
        std_precision = np.std(all_precisions, axis=0)

        upper_bound = mean_precision + std_precision
        lower_bound = mean_precision - std_precision
        
        upper_bound_list.append(upper_bound)
        lower_bound_list.append(lower_bound)

        # Plot shaded area and mean precision line
        ax.fill_between(recall, lower_bound, upper_bound, color=colors[i], alpha=0.3)
        ax.plot(recall, mean_precision, label=labels[i], color=colors[i])

    
    # Add zoomed-in plot
    axins = ax.inset_axes([0.05, 0.05, 0.35, 0.35])
    # Iterate over all curves to add them to the zoomed-in plot
    for i in range(len(recall_list)):
        recall = recall_list[i]
        lower_bound = lower_bound_list[i]
        upper_bound = upper_bound_list[i]
        mean_precision = mean_precision_list[i]
        axins.fill_between(recall, lower_bound, upper_bound, color=colors[i], alpha=0.3)
        axins.plot(recall, mean_precision, label=labels[i], color=colors[i])
    
    axins.set_xlim(0.75, 0.85)
    axins.set_ylim(0.65, 0.95)
    axins.set_xticklabels('')
    axins.set_yticklabels('')
    ax.indicate_inset_zoom(axins)
    # ax.set_title('Precision-Recall Curves by Band')
    ax.set_xlabel('Recall')
    ax.set_ylabel('Precision')
    
    ax.legend()
    ax.set_ylim([0,1.1])
    ax.set_xlim([0,1.2])
    ax.grid(True)

def plot_error_bars(ax, grouped):
    """
    Helper function to plot error bars for COCO bbox mAP metrics by spectral band.

    Parameters:
    ax (matplotlib.axes._subplots.AxesSubplot): The subplot axes to plot on.
    grouped (pandas.DataFrame): DataFrame containing mean and std of coco/bbox mAP metrics by spectral band.
    """
    # Plotting the error bars for each metric
    ax.errorbar(grouped['Band'], grouped['mean_mAP'], yerr=grouped['std_mAP'], 
                 label='$AP$', fmt='-o', capsize=5)

    ax.errorbar(grouped['Band'], grouped['mean_mAP_50'], yerr=grouped['std_mAP_50'], 
                 label='$AP_{50}$', fmt='-s', capsize=5)

    ax.errorbar(grouped['Band'], grouped['mean_mAP_75'], yerr=grouped['std_mAP_75'], 
                 label='$AP_{75}$', fmt='-^', capsize=5)

    # Customizing the plot
    ax.set_xticks(np.arange(1, 13))
    ax.set_xticklabels([f'$B_{{{i}}}$' for i in np.arange(1, 13)])
    ax.set_ylim([0,1.1])
    ax.set_xlabel('Spectral Band')
    ax.set_ylabel('Mean Metric Value')
    # ax.set_title('Error Plot of COCO BBox mAP Metrics by Band')
    ax.legend()
    ax.grid(True)

# Main plotting function
def main(pr_x_band, all_band):
    """
    Main function to generate subplots for precision-recall curves and error bars for COCO bbox mAP metrics.

    Parameters:
    pr_x_band (list): List of evaluation results for different spectral bands.
    all_band (pandas.DataFrame): DataFrame containing coco/bbox mAP metrics grouped by spectral band.
    """
    # Create figure and subplots
    scale_factor = 1.5
    plt.figure(figsize=(15 * scale_factor, 5 * scale_factor))

    # First subplot: Precision-Recall Curves
    ax1 = plt.subplot(1, 2, 1)
    plot_precision_recall(ax1, pr_x_band)

    # Group the data by 'Band' and calculate the mean and standard deviation for second subplot
    grouped = all_band.groupby('Band').agg(
        mean_mAP=('coco/bbox_mAP', 'mean'),
        std_mAP=('coco/bbox_mAP', 'std'),
        mean_mAP_50=('coco/bbox_mAP_50', 'mean'),
        std_mAP_50=('coco/bbox_mAP_50', 'std'),
        mean_mAP_75=('coco/bbox_mAP_75', 'mean'),
        std_mAP_75=('coco/bbox_mAP_75', 'std')
    ).reset_index()

    # Second subplot: Error bars for COCO bbox mAP metrics
    ax2 = plt.subplot(1, 2, 2)
    plot_error_bars(ax2, grouped)
    # Add text (a) and (b) under axes
    ax1.text(0.25, -0.14, '(a) Precision-Recall Curves (VDVRaw)', transform=ax1.transAxes, va='top')
    ax2.text(0.25, -0.14, '(b) Error Plot of BBox Metrics (VDVRaw)', transform=ax2.transAxes, va='top')

    # Adjust layout and display plot
    plt.subplots_adjust(hspace=0.45)
    plt.savefig('VENuS/Venus_PR_curve.png', bbox_inches='tight')
    plt.show()

# Example call to main function (replace with actual data)
main(pr_x_band, all_band_best)

# Multi 

#### Step 1)

In [ ]:
from pathlib import Path
import pandas as pd

from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

from eval import evaluate_object_detector, parse_and_plot, find_json, parse_json_test

VISUALIZE = False

#### Utils
multi_folers = list(Path('/Data_large/marine/PythonProjects/MMDET/checkpoints/Venus/Multi').iterdir())
identifiers = [x.stem.split('perfect_')[-1] for x in multi_folers]


# 1. find coco metrics files
folder = Path('/Data_large/marine/PythonProjects/MMDET/checkpoints/Venus/Multi')
cocometrics = list(folder.glob('**/**/coco_metrics.json'))

# 2. loop over all json files and getting the database
all_band = pd.DataFrame([parse_json_test(x) for x in cocometrics])
all_band.reset_index(inplace=True)
# print(all_band.head())

# 3. Find best config
grouped_bs = all_band.groupby(['Band', 'BS','LR','ME']).mean() # Average of the Seeds:
#   Compute best config:
bestConfig = {idx:[] for idx in identifiers}
for idx, bandSel in enumerate(identifiers):
    BS, LR, ME = grouped_bs.loc[bandSel]['coco/bbox_mAP_50'].idxmax()
    BS, LR, ME = int(BS), float(LR), int(ME)
    bestConfig[bandSel] = (BS, LR, ME)
#   Save
pd.DataFrame(bestConfig).to_csv('Venus/Multi_bestConfig.csv')

# 4. Filtering all results on the best config:
all_band_best = pd.DataFrame()
for Band in identifiers:
    BS, LR, ME = bestConfig[Band]
    filtered_all_band = all_band[
        (all_band['Band'] == Band) &
        (all_band['BS'] == str(BS)) &
        (all_band['LR'] == str(LR)) &
        (all_band['ME'] == str(ME))]
    all_band_best = pd.concat([all_band_best, filtered_all_band])
    
pd.to_pickle(all_band_best, 'Venus/Multi_all_best.pkl')

#### Step 2) Process PR-Curve for each configuration

In [ ]:
pr_x_band = {i:[] for i in identifiers}

grouped_bs = all_band_best.groupby(['Band', 'BS', 'LR', 'ME']).mean() # average over seeds

for idx, row in tqdm(grouped_bs.iterrows()):
    print(f"Band: {idx[0]}, BS: {idx[1]}, LR: {idx[2]}, ME: {idx[3]}, mAP: {row['coco/bbox_mAP_50']:0.2f}")
    Band = idx[0]
    BS = idx[1]
    LR = idx[2]
    ME = idx[3]
    
    bandSelection = idx[0] # all bands
    firstBand = idx[0][1] # bc we use the first band as labels
    ann_file = f'/Data_large/marine/Datasets/VENuS/annotations/perfect/test__band_{firstBand}.json'
    # get folder :
    goTo = f'_{bandSelection}'
    band_path = f'/Data_large/marine/PythonProjects/MMDET/checkpoints/VENuS/Multi/perfect{goTo}'
    folders = list(Path(f'{band_path}').iterdir())  
    # filter folders with the BS
    folders = [f for f in folders if (f'BS_{BS}' in f.name) and (f'LR_{LR}' in f.name) and (f'ME_{ME}' in f.name)]
    
    for folder in folders:
        res_file = list(folder.glob('**/*.json'))
        res_file = [x for x in res_file if 'test_results' in str(x)]
        try:
            assert len(res_file) == 1, f"Found {len(res_file)} json files. Expected 1. \n {res_file}"
        except AssertionError:
            print(f"Error in {res_file}")
            continue
        res_file = res_file[0].as_posix()
        print(f"Res file: {res_file}")
        try:
            stats, eval_results, cocoeval_tmp = evaluate_object_detector(res_file, ann_file)
            pr_x_band[Band].append(cocoeval_tmp)
        except IndexError:
            print(f"Error in {res_file}")
            continue
        
pd.to_pickle(pr_x_band, 'VENuS/Multi_Venus_PR_curve.pkl')


### Plotting

In [ ]:
import pandas as pd
from style import PR

# all_band_best: DataFrame containing coco/bbox mAP metrics grouped by spectral band
# pr_x_band: List of evaluation results for each band
all_band_best = pd.read_pickle('VENuS/Multi_all_best.pkl')
pr_x_band = pd.read_pickle('VENuS/Multi_Venus_PR_curve.pkl')

# Example call to main function:
g = PR(pr_x_band, all_band_best, savepath='VENuS/Multi_Venus_PR_curve.png')

In [ ]:
g

In [ ]:
g = g.loc[:, ~g.columns.str.contains('std')]

In [ ]:
g

In [ ]:
from style import reformat
g['Band'] = g['Band'].apply(reformat)

In [ ]:
g.to_csv('VENuS/Multi_table.csv')

In [ ]:
g

# Single v2

In [5]:
from pathlib import Path
import pandas as pd
from tqdm import tqdm
import warnings
import json
warnings.filterwarnings('ignore')

from eval import evaluate_object_detector, parse_and_plot, find_json


class resultHandler():
    def __init__(self, folder_path):
        self.folder_path = folder_path
        self.identifiers = [x.stem.split('perfect_')[-1] for x in self.folder_path.iterdir()]
    
    def find_coco_metrics_files(self):
        cocometrics = list(self.folder_path.glob('**/**/coco_metrics.json'))
        return cocometrics
    
    def parse_json_test(self, json_file_path):
        # implementation of parse_json_test function
        partial = {
            'Seed': Path(json_file_path).parent.parent.name.split('_')[0],
            'BS': Path(json_file_path).parent.parent.name.split('_')[2],
            'LR': Path(json_file_path).parent.parent.name.split('_')[4],
            'ME': Path(json_file_path).parent.parent.name.split('_')[6],
            'OPT': Path(json_file_path).parent.parent.name.split('_')[8],
            'Band': Path(json_file_path).parent.parent.parent.name.split('perfect_')[-1],
        }

        # Read the JSON file
        with open(json_file_path, 'r') as f:
            data = json.load(f)


        # merge data and partial 
        data = {**data, **partial}
        return data
    
    def find_best_config(self, all_band):
        grouped_bs = all_band.groupby(['Band', 'BS', 'LR', 'ME']).mean()
        bestConfig = {idx: [] for idx in self.identifiers}
        for idx, bandSel in enumerate(self.identifiers):
            BS, LR, ME = grouped_bs.loc[bandSel]['coco/bbox_mAP_50'].idxmax()
            BS, LR, ME = int(BS), float(LR), int(ME)
            bestConfig[bandSel] = (BS, LR, ME)
        return bestConfig
    
    def filter_results(self, all_band, bestConfig):
        all_band_best = pd.DataFrame()
        for Band in self.identifiers:
            BS, LR, ME = bestConfig[Band]
            filtered_all_band = all_band[
                (all_band['Band'] == Band) &
                (all_band['BS'] == str(BS)) &
                (all_band['LR'] == str(LR)) &
                (all_band['ME'] == str(ME))]
            all_band_best = pd.concat([all_band_best, filtered_all_band])
        return all_band_best
    
    def process_results(self):
        cocometrics = self.find_coco_metrics_files()
        all_band = pd.DataFrame([self.parse_json_test(x) for x in cocometrics])
        all_band.reset_index(inplace=True)
        bestConfig = self.find_best_config(all_band)
        all_band_best = self.filter_results(all_band, bestConfig)
        return all_band_best


# Create an instance of the resultHandler class
mode = 'Single' # 'Single' or 'Multi'
Sat = 'VENuS' # 'VENuS' or 'Sentinel'
handler = resultHandler(Path(f'/Data_large/marine/PythonProjects/MMDET/checkpoints/{Sat}/{mode}'))

# Process the results
all_band_best = handler.process_results()

# Save the processed results
pd.to_pickle(all_band_best, f'{Sat}/{mode}_all_best.pkl')

In [ ]:
pr_x_band = {i:[] for i in identifiers}

grouped_bs = all_band_best.groupby(['Band', 'BS', 'LR', 'ME']).mean() # average over seeds

for idx, row in tqdm(grouped_bs.iterrows()):
    print(f"Band: {idx[0]}, BS: {idx[1]}, LR: {idx[2]}, ME: {idx[3]}, mAP: {row['coco/bbox_mAP_50']:0.2f}")
    Band = idx[0]
    BS = idx[1]
    LR = idx[2]
    ME = idx[3]
    
    bandSelection = idx[0] # all bands
    firstBand = Band.split('b')[-1] # bc we use the first band as labels
    ann_file = f'/Data_large/marine/Datasets/VENuS/annotations/perfect/test__band_{firstBand}.json'
    # get folder :
    goTo = f'_{bandSelection}'
    band_path = f'/Data_large/marine/PythonProjects/MMDET/checkpoints/VENuS/Single/perfect{goTo}'
    folders = list(Path(f'{band_path}').iterdir())  
    # filter folders with the BS
    folders = [f for f in folders if (f'BS_{BS}' in f.name) and (f'LR_{LR}' in f.name) and (f'ME_{ME}' in f.name)]
    
    for folder in folders:
        res_file = list(folder.glob('**/*.json'))
        res_file = [x for x in res_file if 'test_results' in str(x)]
        try:
            assert len(res_file) == 1, f"Found {len(res_file)} json files. Expected 1. \n {res_file}"
        except AssertionError:
            print(f"Error in {res_file}")
            continue
        res_file = res_file[0].as_posix()
        print(f"Res file: {res_file}")
        try:
            stats, eval_results, cocoeval_tmp = evaluate_object_detector(res_file, ann_file)
            pr_x_band[Band].append(cocoeval_tmp)
        except IndexError:
            print(f"Error in {res_file}")
            continue
        
pd.to_pickle(pr_x_band, 'VENuS/Single_Venus_PR_curve.pkl')


In [ ]:
import pandas as pd
from style import PR

# all_band_best: DataFrame containing coco/bbox mAP metrics grouped by spectral band
# pr_x_band: List of evaluation results for each band
all_band_best = pd.read_pickle('VENuS/Single_all_best.pkl')
pr_x_band = pd.read_pickle('VENuS/Single_Venus_PR_curve.pkl')

# Verifica che pr_x_band non sia vuoto
if not pr_x_band:
    raise ValueError("La lista pr_x_band è vuota. Assicurati che il file 'VENuS/Single_Venus_PR_curve.pkl' contenga dati validi.")

# Verifica che all_band_best non sia vuoto
if all_band_best.empty:
    raise ValueError("Il DataFrame all_band_best è vuoto. Assicurati che il file 'VENuS/Single_all_best.pkl' contenga dati validi.")

# Example call to main function:
g = PR(pr_x_band, all_band_best, savepath='VENuS/Single_Venus_PR_curve.png',)

# Sentinel Single

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import pandas as pd


from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

from eval import evaluate_object_detector, parse_and_plot, find_json, parse_json_test

#### Utils
multi_folers = list(Path('/Data_large/marine/PythonProjects/MMDET/checkpoints/Sentinel/Single').iterdir())
identifiers = [x.stem.split('perfect_')[-1] for x in multi_folers]


# 1. find coco metrics files
folder = Path('/Data_large/marine/PythonProjects/MMDET/checkpoints/Sentinel/Single')
cocometrics = list(folder.glob('**/**/coco_metrics.json'))

# 2. loop over all json files and getting the database
all_band = pd.DataFrame([parse_json_test(x) for x in cocometrics])
all_band.reset_index(inplace=True)
# print(all_band.head())

# 3. Find best config
grouped_bs = all_band.groupby(['Band', 'BS','LR','ME']).mean() # Average of the Seeds:
#   Compute best config:
bestConfig = {idx:[] for idx in identifiers}
for idx, bandSel in enumerate(identifiers):
    BS, LR, ME = grouped_bs.loc[bandSel]['coco/bbox_mAP_50'].idxmax()
    BS, LR, ME = int(BS), float(LR), int(ME)
    bestConfig[bandSel] = (BS, LR, ME)

# Hook for the best config: taking into account NaN values    
bestConfig['b8'] = (2, 0.001, 130)

#   Save
pd.DataFrame(bestConfig).to_csv('Sentinel/Single_bestConfig.csv')

# 4. Filtering all results on the best config:
all_band_best = pd.DataFrame()
for Band in identifiers:
    BS, LR, ME = bestConfig[Band]
    filtered_all_band = all_band[
        (all_band['Band'] == Band) &
        (all_band['BS'] == str(BS)) &
        (all_band['LR'] == str(LR)) &
        (all_band['ME'] == str(ME))]
    all_band_best = pd.concat([all_band_best, filtered_all_band])
    
pd.to_pickle(all_band_best, 'Sentinel/Single_all_best.pkl')

In [ ]:
import logging

# Create a logger
logger = logging.getLogger(__name__)

# Set the logging level
logger.setLevel(logging.INFO)

# Create a file handler
file_handler = logging.FileHandler('Sentinel/Sentinel_single_PR.log')

# Create a formatter
formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')

# Set the formatter for the file handler
file_handler.setFormatter(formatter)

# Add the file handler to the logger
logger.addHandler(file_handler)

# Log a message
logger.info('Logger is set up')

In [ ]:
pr_x_band = {i:[] for i in identifiers}

grouped_bs = all_band_best.groupby(['Band', 'BS', 'LR', 'ME']).mean() # average over seeds

for idx, row in tqdm(grouped_bs.iterrows()):
    print(f"Band: {idx[0]}, BS: {idx[1]}, LR: {idx[2]}, ME: {idx[3]}, mAP: {row['coco/bbox_mAP_50']:0.2f}")
    Band = idx[0]
    BS = idx[1]
    LR = idx[2]
    ME = idx[3]
    
    bandSelection = idx[0] # all bands
    firstBand = Band.split('b')[-1] # bc we use the first band as labels
    ann_file = f'/Data_large/marine/Datasets/VDS2Raw/annotations/test__band_{firstBand}.json'
    # get folder :
    goTo = f'_{bandSelection}'
    band_path = f'/Data_large/marine/PythonProjects/MMDET/checkpoints/Sentinel/Single/perfect{goTo}'
    folders = list(Path(f'{band_path}').iterdir())  
    # filter folders with the BS
    folders = [f for f in folders if (f'BS_{BS}' in f.name) and (f'LR_{LR}' in f.name) and (f'ME_{ME}' in f.name)]
    
    for folder in folders:
        res_file = list(folder.glob('**/*.json'))
        res_file = [x for x in res_file if 'test_results' in str(x)]
        try:
            assert len(res_file) == 1, f"Found {len(res_file)} json files. Expected 1. \n {res_file}"
        except AssertionError:
            print(f"Error in {res_file}")
            continue
        res_file = res_file[0].as_posix()
        print(f"Res file: {res_file}")
        try:
            stats, eval_results, cocoeval_tmp = evaluate_object_detector(res_file, ann_file)
            pr_x_band[Band].append(cocoeval_tmp)
        except IndexError:
            print(f"Error in {res_file}")
            logger.error(f"Error in {res_file}")
            continue
        
pd.to_pickle(pr_x_band, 'Sentinel/Single_Sentinel_PR_curve.pkl')


In [ ]:
import pandas as pd
from style import PR

# all_band_best = pd.read_pickle('VENuS/Single_all_best.pkl')
# pr_x_band = pd.read_pickle('VENuS/Single_Venus_PR_curve.pkl')

# Example call to main function:
g = PR(pr_x_band, all_band_best, savepath='Sentinel/Single_Venus_PR_curve.png',)

In [ ]:
g.to_csv('Sentinel/Single_table.csv')

# Multi Sentinel

In [ ]:
from pathlib import Path
import pandas as pd


from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

from eval import evaluate_object_detector, parse_and_plot, find_json, parse_json_test


#### Utils
multi_folers = list(Path('/Data_large/marine/PythonProjects/MMDET/checkpoints/Sentinel/Multi').iterdir())
identifiers = [x.stem.split('perfect_')[-1] for x in multi_folers]


# 1. find coco metrics files
folder = Path('/Data_large/marine/PythonProjects/MMDET/checkpoints/Sentinel/Multi')
cocometrics = list(folder.glob('**/**/coco_metrics.json'))

# 2. loop over all json files and getting the database
all_band = pd.DataFrame([parse_json_test(x) for x in cocometrics])
all_band.reset_index(inplace=True)
# print(all_band.head())

# 3. Find best config
grouped_bs = all_band.groupby(['Band', 'BS','LR','ME']).mean() # Average of the Seeds:
#   Compute best config:
bestConfig = {idx:[] for idx in identifiers}
for idx, bandSel in enumerate(identifiers):
    BS, LR, ME = grouped_bs.loc[bandSel]['coco/bbox_mAP_50'].idxmax()
    BS, LR, ME = int(BS), float(LR), int(ME)
    bestConfig[bandSel] = (BS, LR, ME)
#   Save
pd.DataFrame(bestConfig).to_csv('Sentinel/Multi_bestConfig.csv')

# 4. Filtering all results on the best config:
all_band_best = pd.DataFrame()
for Band in identifiers:
    BS, LR, ME = bestConfig[Band]
    filtered_all_band = all_band[
        (all_band['Band'] == Band) &
        (all_band['BS'] == str(BS)) &
        (all_band['LR'] == str(LR)) &
        (all_band['ME'] == str(ME))]
    all_band_best = pd.concat([all_band_best, filtered_all_band])
    
pd.to_pickle(all_band_best, 'Sentinel/Single_all_best.pkl')

In [ ]:
pr_x_band = {i:[] for i in identifiers}

grouped_bs = all_band_best.groupby(['Band', 'BS', 'LR', 'ME']).mean() # average over seeds

for idx, row in tqdm(grouped_bs.iterrows()):
    print(f"Band: {idx[0]}, BS: {idx[1]}, LR: {idx[2]}, ME: {idx[3]}, mAP: {row['coco/bbox_mAP_50']:0.2f}")
    Band = idx[0]
    BS = idx[1]
    LR = idx[2]
    ME = idx[3]
    
    bandSelection = idx[0] # all bands
    firstBand = Band.split('b')[-1] # bc we use the first band as labels
    ann_file = f'/Data_large/marine/Datasets/VDS2Raw/annotations/test__band_{firstBand}.json'
    # get folder :
    goTo = f'_{bandSelection}'
    band_path = f'/Data_large/marine/PythonProjects/MMDET/checkpoints/Sentinel/Multi/perfect{goTo}'
    folders = list(Path(f'{band_path}').iterdir())  
    # filter folders with the BS
    folders = [f for f in folders if (f'BS_{BS}' in f.name) and (f'LR_{LR}' in f.name) and (f'ME_{ME}' in f.name)]
    
    for folder in folders:
        res_file = list(folder.glob('**/*.json'))
        res_file = [x for x in res_file if 'test_results' in str(x)]
        try:
            assert len(res_file) == 1, f"Found {len(res_file)} json files. Expected 1. \n {res_file}"
        except AssertionError:
            print(f"Error in {res_file}")
            continue
        res_file = res_file[0].as_posix()
        print(f"Res file: {res_file}")
        try:
            stats, eval_results, cocoeval_tmp = evaluate_object_detector(res_file, ann_file)
            pr_x_band[Band].append(cocoeval_tmp)
        except IndexError:
            print(f"Error in {res_file}")
            logger.error(f"Error in {res_file}")
            continue
        
pd.to_pickle(pr_x_band, 'Sentinel/Multi_Sentinel_PR_curve.pkl')


In [ ]:
import pandas as pd
from style import PR

# all_band_best = pd.read_pickle('VENuS/Single_all_best.pkl')
# pr_x_band = pd.read_pickle('VENuS/Single_Venus_PR_curve.pkl')

# Example call to main function:
g = PR(pr_x_band, all_band_best, savepath='Sentinel/Multi_Venus_PR_curve.png',)
g.to_csv('Sentinel/Multi_table.csv')

g